# Aether Stage 3 training (Colab A100)

Clones the repo and runs the existing scripts from `scripts/` cell by cell — no logic duplicated here. Uses Colab's own Python environment directly (it already ships a CUDA-matched torch for the A100), no separate venv/uv layer. Run cells top to bottom.

In [ ]:
!git clone https://github.com/karl4th/aether.git
%cd aether

## Environment setup

Torch/torchaudio are left alone (Colab's preinstalled build already matches the A100's driver). Only the extra packages are installed. `onnxruntime` (CPU-only, pulled in by `piper-tts`) is swapped for `onnxruntime-gpu` so Piper TTS synthesis runs on the GPU too instead of bottlenecking on CPU.

In [ ]:
%pip install -q transformers accelerate datasets piper-tts soundfile pyyaml tqdm
%pip uninstall -y -q onnxruntime
%pip install -q onnxruntime-gpu

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Stage 2: build the dataset (20,000 sentences)

Each script skips files that already exist by id, so re-running any cell after a disconnect is safe and resumes instead of restarting from zero.

In [ ]:
!python -m piper.download_voices en_US-lessac-medium --download-dir data/piper_voices

In [ ]:
!python scripts/build_sentences.py --num-sentences 20000

In [ ]:
!python scripts/synthesize_tts.py --use-cuda

In [ ]:
!python scripts/extract_hidden_states.py

## Stage 3: train the decoder

`--val-size 1000` keeps roughly the same ~5% held-out fraction as the earlier 1500-sentence run. Checkpoints (`best.pt`, periodic `epoch_NNNN.pt`, `last.pt`) are written to `checkpoints/decoder/` as training goes. Colab's local disk doesn't survive a runtime reset, so if the session might disconnect for a while, mount Drive first and point `--checkpoint-dir` there (or just re-run the zip/download cell at the bottom periodically).

In [ ]:
!python scripts/train_decoder.py --epochs 50 --val-size 1000

## Listen to a result

In [ ]:
!python scripts/listen_to_decoder.py --id 00042

In [ ]:
from IPython.display import Audio, display

print("Predicted:")
display(Audio("data/samples/00042.wav"))
print("Ground truth (Piper):")
display(Audio("data/dataset/audio/00042.wav"))

## Download the trained checkpoint

In [ ]:
!zip -rq decoder_checkpoints.zip checkpoints/decoder
from google.colab import files
files.download("decoder_checkpoints.zip")